In [ ]:
!pip install -q transformers datasets accelerate sentencepiece seqeval openpyxl evaluate

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving pershop_slots.xlsx to pershop_slots.xlsx


In [ ]:
import pandas as pd

df = pd.read_excel("pershop_slots.xlsx")
df = df.dropna(subset=["annotated_text"]).reset_index(drop=True)
print(df.head())
print(len(df))


                                      annotated_text
0                                          سلام خوبی
1                                              ممنون
2                  من یک عدد [شیر](product) می خواهم
3  [شیر](product) مخصوص [خردسالان](cat) [ماجان](b...
4                                         نه نمیخوام
11065


In [ ]:
import re

slot_pattern = re.compile(r'\[([^\]]+)\]\(([^)]+)\)')

def parse_annotated(annotated_text):
    """
    ورودی: رشته‌ای مثل
    "من یه [شیر](product) [کم چرب](fat) می‌خوام"
    خروجی: tokens, tags (لیست توکن و لیبل BIO)
    """
    tokens = []
    tags = []
    idx = 0
    text = annotated_text

    while idx < len(text):
        m = slot_pattern.search(text, idx)
        if not m:
            # بخش باقی‌مانده بدون اسلات
            remaining = text[idx:].strip()
            if remaining:
                for tok in remaining.split():
                    tokens.append(tok)
                    tags.append("O")
            break

        # متن قبل از اسلات (non-annotated span)
        pre = text[idx:m.start()]
        pre = pre.strip()
        if pre:
            for tok in pre.split():
                tokens.append(tok)
                tags.append("O")

        span_text = m.group(1).strip()    # متن داخل []
        slot_name = m.group(2).strip()    # نام اسلات داخل ()

        span_tokens = span_text.split()
        if len(span_tokens) == 1:
            tokens.append(span_tokens[0])
            tags.append(f"B-{slot_name}")
        else:
            for j, tok in enumerate(span_tokens):
                tokens.append(tok)
                if j == 0:
                    tags.append(f"B-{slot_name}")
                else:
                    tags.append(f"I-{slot_name}")

        idx = m.end()

    return tokens, tags

# تست روی چند نمونه
for i in range(3):
    t, y = parse_annotated(df.iloc[i]["annotated_text"])
    print(df.iloc[i]["annotated_text"])
    print(list(zip(t, y)))
    print("----")


سلام خوبی
[('سلام', 'O'), ('خوبی', 'O')]
----
ممنون
[('ممنون', 'O')]
----
من یک عدد [شیر](product) می خواهم
[('من', 'O'), ('یک', 'O'), ('عدد', 'O'), ('شیر', 'B-product'), ('می', 'O'), ('خواهم', 'O')]
----


In [ ]:
from sklearn.model_selection import train_test_split

records = []

for _, row in df.iterrows():
    tokens, tags = parse_annotated(row["annotated_text"])
    if not tokens:
        continue
    records.append({
        "tokens": tokens,
        "tags": tags
    })

slot_df = pd.DataFrame(records)
print(slot_df.head())
print(len(slot_df))


                                              tokens  \
0                                       [سلام, خوبی]   
1                                            [ممنون]   
2                      [من, یک, عدد, شیر, می, خواهم]   
3  [شیر, مخصوص, خردسالان, ماجان, 200, میلی, لیتری...   
4                                      [نه, نمیخوام]   

                                                tags  
0                                             [O, O]  
1                                                [O]  
2                         [O, O, O, B-product, O, O]  
3  [B-product, O, B-cat, B-brand_2, B-volume, I-v...  
4                                             [O, O]  
11065


In [ ]:
train_df, test_df = train_test_split(slot_df, test_size=0.2, random_state=42)
train_df, dev_df  = train_test_split(train_df, test_size=0.125, random_state=42)
print(len(train_df), len(dev_df), len(test_df))


7745 1107 2213


In [ ]:
from datasets import Dataset, DatasetDict

train_ds = Dataset.from_pandas(train_df)
dev_ds   = Dataset.from_pandas(dev_df)
test_ds  = Dataset.from_pandas(test_df)

dataset = DatasetDict({
    "train": train_ds,
    "validation": dev_ds,
    "test": test_ds
})

dataset


DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags', '__index_level_0__'],
        num_rows: 7745
    })
    validation: Dataset({
        features: ['tokens', 'tags', '__index_level_0__'],
        num_rows: 1107
    })
    test: Dataset({
        features: ['tokens', 'tags', '__index_level_0__'],
        num_rows: 2213
    })
})

In [ ]:
all_tags = set()

for split in ["train", "validation", "test"]:
    for example in dataset[split]:
        for tag in example["tags"]:
            all_tags.add(tag)

all_tags = sorted(all_tags)
print(all_tags)

label2id = {label: i for i, label in enumerate(all_tags)}
id2label = {i: label for label, i in label2id.items()}

label2id


['B-brand', 'B-brand_2', 'B-brew-method', 'B-capacity', 'B-cat', 'B-charbi', 'B-cheese_shape', 'B-color', 'B-country', 'B-gazdar', 'B-gender', 'B-herb-content', 'B-length', 'B-material', 'B-package', 'B-product', 'B-refresh', 'B-size', 'B-size_number', 'B-taste', 'B-type', 'B-unit-count', 'B-volume', 'B-weight', 'I-brand', 'I-brew-method', 'I-capacity', 'I-cat', 'I-charbi', 'I-cheese_shape', 'I-color', 'I-gazdar', 'I-herb-content', 'I-length', 'I-material', 'I-package', 'I-product', 'I-refresh', 'I-size', 'I-taste', 'I-type', 'I-unit-count', 'I-volume', 'I-weight', 'O']


{'B-brand': 0,
 'B-brand_2': 1,
 'B-brew-method': 2,
 'B-capacity': 3,
 'B-cat': 4,
 'B-charbi': 5,
 'B-cheese_shape': 6,
 'B-color': 7,
 'B-country': 8,
 'B-gazdar': 9,
 'B-gender': 10,
 'B-herb-content': 11,
 'B-length': 12,
 'B-material': 13,
 'B-package': 14,
 'B-product': 15,
 'B-refresh': 16,
 'B-size': 17,
 'B-size_number': 18,
 'B-taste': 19,
 'B-type': 20,
 'B-unit-count': 21,
 'B-volume': 22,
 'B-weight': 23,
 'I-brand': 24,
 'I-brew-method': 25,
 'I-capacity': 26,
 'I-cat': 27,
 'I-charbi': 28,
 'I-cheese_shape': 29,
 'I-color': 30,
 'I-gazdar': 31,
 'I-herb-content': 32,
 'I-length': 33,
 'I-material': 34,
 'I-package': 35,
 'I-product': 36,
 'I-refresh': 37,
 'I-size': 38,
 'I-taste': 39,
 'I-type': 40,
 'I-unit-count': 41,
 'I-volume': 42,
 'I-weight': 43,
 'O': 44}

PARSBERT for token Classfication

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "HooshvareLab/bert-base-parsbert-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

num_labels = len(all_tags)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)


config.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at HooshvareLab/bert-base-parsbert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
max_length = 128

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

    all_labels = []
    for i, word_ids in enumerate(tokenized.word_ids(batch_index=j) for j in range(len(examples["tokens"]))):
        word_ids = list(word_ids)
        labels = []
        tags = examples["tags"][i]
        previous_word_id = None
        for w_id in word_ids:
            if w_id is None:
                labels.append(-100)  # نادیده در loss
            else:
                if w_id != previous_word_id:
                    labels.append(label2id[tags[w_id]])
                else:
                    # ادامهٔ subword
                    labels.append(label2id[tags[w_id]])
                previous_word_id = w_id
        all_labels.append(labels)

    tokenized["labels"] = all_labels
    return tokenized

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["tokens", "tags"])
tokenized_datasets.set_format(type="torch")
tokenized_datasets


Map:   0%|          | 0/7745 [00:00<?, ? examples/s]

Map:   0%|          | 0/1107 [00:00<?, ? examples/s]

Map:   0%|          | 0/2213 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 7745
    })
    validation: Dataset({
        features: ['__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1107
    })
    test: Dataset({
        features: ['__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2213
    })
})

In [ ]:
from evaluate import load as load_eval

seqeval = load_eval("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)

    true_labels = []
    true_preds = []

    for pred, lab in zip(predictions, labels):
        curr_true_labels = []
        curr_true_preds = []
        for p, l in zip(pred, lab):
            if l == -100:
                continue
            curr_true_labels.append(id2label[l])
            curr_true_preds.append(id2label[p])
        true_labels.append(curr_true_labels)
        true_preds.append(curr_true_preds)

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    # F1 کلی
    return {"f1": results["overall_f1"], "precision": results["overall_precision"], "recall": results["overall_recall"]}

In [ ]:
import os
os.environ["ACCELERATE_DISABLE_RICH"] = "1"

from transformers import TrainingArguments, Trainer

batch_size = 16

training_args = TrainingArguments(
    output_dir="./pershop_slots_parsbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    disable_tqdm=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipython-input-3934561526.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


{'loss': 0.0374, 'grad_norm': 0.047664839774370193, 'learning_rate': 1.9326460481099656e-05, 'epoch': 0.10309278350515463}
{'loss': 0.0858, 'grad_norm': 9.648211479187012, 'learning_rate': 1.863917525773196e-05, 'epoch': 0.20618556701030927}
{'loss': 0.0629, 'grad_norm': 2.9911656379699707, 'learning_rate': 1.7951890034364265e-05, 'epoch': 0.30927835051546393}
{'loss': 0.0727, 'grad_norm': 3.0430123805999756, 'learning_rate': 1.7264604810996566e-05, 'epoch': 0.41237113402061853}
{'loss': 0.0717, 'grad_norm': 0.10301941633224487, 'learning_rate': 1.657731958762887e-05, 'epoch': 0.5154639175257731}
{'loss': 0.0783, 'grad_norm': 5.7426042556762695, 'learning_rate': 1.5890034364261167e-05, 'epoch': 0.6185567010309279}
{'loss': 0.058, 'grad_norm': 4.12587833404541, 'learning_rate': 1.5202749140893472e-05, 'epoch': 0.7216494845360825}
{'loss': 0.0571, 'grad_norm': 0.6340067386627197, 'learning_rate': 1.4515463917525774e-05, 'epoch': 0.8247422680412371}
{'loss': 0.0564, 'grad_norm': 0.0757280

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.09704338759183884, 'eval_f1': 0.9435718440954044, 'eval_precision': 0.9321839080459771, 'eval_recall': 0.9552414605418139, 'eval_runtime': 8.0398, 'eval_samples_per_second': 137.69, 'eval_steps_per_second': 8.707, 'epoch': 2.0}
{'loss': 0.0216, 'grad_norm': 4.324546813964844, 'learning_rate': 6.268041237113403e-06, 'epoch': 2.0618556701030926}
{'loss': 0.0113, 'grad_norm': 0.007368447259068489, 'learning_rate': 5.580756013745705e-06, 'epoch': 2.1649484536082473}
{'loss': 0.0103, 'grad_norm': 0.061456575989723206, 'learning_rate': 4.893470790378007e-06, 'epoch': 2.268041237113402}
{'loss': 0.0142, 'grad_norm': 0.09221767634153366, 'learning_rate': 4.2061855670103096e-06, 'epoch': 2.3711340206185567}
{'loss': 0.0146, 'grad_norm': 0.033031780272722244, 'learning_rate': 3.518900343642612e-06, 'epoch': 2.4742268041237114}
{'loss': 0.0153, 'grad_norm': 0.2163134664297104, 'learning_rate': 2.831615120274914e-06, 'epoch': 2.5773195876288657}
{'loss': 0.0066, 'grad_norm': 0.0027

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.10447707772254944, 'eval_f1': 0.9430232558139534, 'eval_precision': 0.931113662456946, 'eval_recall': 0.9552414605418139, 'eval_runtime': 7.993, 'eval_samples_per_second': 138.496, 'eval_steps_per_second': 8.758, 'epoch': 3.0}
{'train_runtime': 580.6697, 'train_samples_per_second': 40.014, 'train_steps_per_second': 2.506, 'train_loss': 0.03525099399262764, 'epoch': 3.0}


TrainOutput(global_step=1455, training_loss=0.03525099399262764, metrics={'train_runtime': 580.6697, 'train_samples_per_second': 40.014, 'train_steps_per_second': 2.506, 'train_loss': 0.03525099399262764, 'epoch': 3.0})

In [ ]:
test_results = trainer.evaluate(tokenized_datasets["test"])
test_results

{'eval_loss': 0.12447374314069748, 'eval_f1': 0.9271300448430494, 'eval_precision': 0.920935412026726, 'eval_recall': 0.9334085778781038, 'eval_runtime': 16.9059, 'eval_samples_per_second': 130.901, 'eval_steps_per_second': 8.222, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.12447374314069748,
 'eval_f1': 0.9271300448430494,
 'eval_precision': 0.920935412026726,
 'eval_recall': 0.9334085778781038,
 'eval_runtime': 16.9059,
 'eval_samples_per_second': 130.901,
 'eval_steps_per_second': 8.222,
 'epoch': 3.0}

XLM-R for token Classfication

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

num_labels = len(all_tags)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
max_length = 128

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

    all_labels = []
    for i, word_ids in enumerate(tokenized.word_ids(batch_index=j) for j in range(len(examples["tokens"]))):
        word_ids = list(word_ids)
        labels = []
        tags = examples["tags"][i]
        previous_word_id = None
        for w_id in word_ids:
            if w_id is None:
                labels.append(-100)  # نادیده در loss
            else:
                if w_id != previous_word_id:
                    labels.append(label2id[tags[w_id]])
                else:
                    # ادامهٔ subword
                    labels.append(label2id[tags[w_id]])
                previous_word_id = w_id
        all_labels.append(labels)

    tokenized["labels"] = all_labels
    return tokenized

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["tokens", "tags"])
tokenized_datasets.set_format(type="torch")
tokenized_datasets


Map:   0%|          | 0/7745 [00:00<?, ? examples/s]

Map:   0%|          | 0/1107 [00:00<?, ? examples/s]

Map:   0%|          | 0/2213 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 7745
    })
    validation: Dataset({
        features: ['__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1107
    })
    test: Dataset({
        features: ['__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2213
    })
})

In [ ]:
from evaluate import load as load_eval

seqeval = load_eval("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)

    true_labels = []
    true_preds = []

    for pred, lab in zip(predictions, labels):
        curr_true_labels = []
        curr_true_preds = []
        for p, l in zip(pred, lab):
            if l == -100:
                continue
            curr_true_labels.append(id2label[l])
            curr_true_preds.append(id2label[p])
        true_labels.append(curr_true_labels)
        true_preds.append(curr_true_preds)

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    # F1 کلی
    return {"f1": results["overall_f1"], "precision": results["overall_precision"], "recall": results["overall_recall"]}

In [ ]:
import os
os.environ["ACCELERATE_DISABLE_RICH"] = "1"

from transformers import TrainingArguments, Trainer

batch_size = 16

training_args = TrainingArguments(
    output_dir="./pershop_slots_XLMR",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    disable_tqdm=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipython-input-1777911920.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


{'loss': 1.8121, 'grad_norm': 8.379809379577637, 'learning_rate': 1.9326460481099656e-05, 'epoch': 0.10309278350515463}
{'loss': 1.0271, 'grad_norm': 13.77624797821045, 'learning_rate': 1.863917525773196e-05, 'epoch': 0.20618556701030927}
{'loss': 0.8373, 'grad_norm': 6.923415184020996, 'learning_rate': 1.7951890034364265e-05, 'epoch': 0.30927835051546393}
{'loss': 0.7457, 'grad_norm': 14.035131454467773, 'learning_rate': 1.7264604810996566e-05, 'epoch': 0.41237113402061853}
{'loss': 0.6526, 'grad_norm': 19.331090927124023, 'learning_rate': 1.657731958762887e-05, 'epoch': 0.5154639175257731}
{'loss': 0.6331, 'grad_norm': 11.126033782958984, 'learning_rate': 1.5890034364261167e-05, 'epoch': 0.6185567010309279}
{'loss': 0.5298, 'grad_norm': 12.54867172241211, 'learning_rate': 1.5202749140893472e-05, 'epoch': 0.7216494845360825}
{'loss': 0.5274, 'grad_norm': 11.874646186828613, 'learning_rate': 1.4515463917525774e-05, 'epoch': 0.8247422680412371}
{'loss': 0.4647, 'grad_norm': 6.4833149909

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.3552718758583069, 'eval_f1': 0.7176966292134831, 'eval_precision': 0.6976109215017064, 'eval_recall': 0.7389732465654375, 'eval_runtime': 7.2181, 'eval_samples_per_second': 153.364, 'eval_steps_per_second': 9.698, 'epoch': 1.0}
{'loss': 0.4572, 'grad_norm': 16.899690628051758, 'learning_rate': 1.314089347079038e-05, 'epoch': 1.0309278350515463}
{'loss': 0.3581, 'grad_norm': 19.721019744873047, 'learning_rate': 1.2453608247422682e-05, 'epoch': 1.134020618556701}
{'loss': 0.386, 'grad_norm': 16.48703956604004, 'learning_rate': 1.1766323024054984e-05, 'epoch': 1.2371134020618557}
{'loss': 0.3143, 'grad_norm': 12.492369651794434, 'learning_rate': 1.1079037800687287e-05, 'epoch': 1.3402061855670104}
{'loss': 0.3314, 'grad_norm': 17.062904357910156, 'learning_rate': 1.039175257731959e-05, 'epoch': 1.443298969072165}
{'loss': 0.336, 'grad_norm': 10.17039680480957, 'learning_rate': 9.70446735395189e-06, 'epoch': 1.5463917525773194}
{'loss': 0.3084, 'grad_norm': 11.3763074874877

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.21923068165779114, 'eval_f1': 0.8257926612041326, 'eval_precision': 0.8139044943820225, 'eval_recall': 0.8380332610267535, 'eval_runtime': 7.2199, 'eval_samples_per_second': 153.325, 'eval_steps_per_second': 9.695, 'epoch': 2.0}
{'loss': 0.3075, 'grad_norm': 23.923599243164062, 'learning_rate': 6.268041237113403e-06, 'epoch': 2.0618556701030926}
{'loss': 0.2671, 'grad_norm': 16.150344848632812, 'learning_rate': 5.580756013745705e-06, 'epoch': 2.1649484536082473}
{'loss': 0.2119, 'grad_norm': 16.917564392089844, 'learning_rate': 4.893470790378007e-06, 'epoch': 2.268041237113402}
{'loss': 0.2255, 'grad_norm': 19.799718856811523, 'learning_rate': 4.2061855670103096e-06, 'epoch': 2.3711340206185567}
{'loss': 0.2097, 'grad_norm': 25.03011703491211, 'learning_rate': 3.518900343642612e-06, 'epoch': 2.4742268041237114}
{'loss': 0.2151, 'grad_norm': 19.173879623413086, 'learning_rate': 2.831615120274914e-06, 'epoch': 2.5773195876288657}
{'loss': 0.1578, 'grad_norm': 10.787364959

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.18505841493606567, 'eval_f1': 0.8594806118818925, 'eval_precision': 0.84593837535014, 'eval_recall': 0.8734634851771511, 'eval_runtime': 7.2252, 'eval_samples_per_second': 153.214, 'eval_steps_per_second': 9.688, 'epoch': 3.0}
{'train_runtime': 1101.9225, 'train_samples_per_second': 21.086, 'train_steps_per_second': 1.32, 'train_loss': 0.4459791892787435, 'epoch': 3.0}


TrainOutput(global_step=1455, training_loss=0.4459791892787435, metrics={'train_runtime': 1101.9225, 'train_samples_per_second': 21.086, 'train_steps_per_second': 1.32, 'train_loss': 0.4459791892787435, 'epoch': 3.0})

In [ ]:
test_results = trainer.evaluate(tokenized_datasets["test"])
test_results

{'eval_loss': 0.21845778822898865, 'eval_f1': 0.8559176672384219, 'eval_precision': 0.8489282068730861, 'eval_recall': 0.8630231753718437, 'eval_runtime': 15.4829, 'eval_samples_per_second': 142.932, 'eval_steps_per_second': 8.978, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.21845778822898865,
 'eval_f1': 0.8559176672384219,
 'eval_precision': 0.8489282068730861,
 'eval_recall': 0.8630231753718437,
 'eval_runtime': 15.4829,
 'eval_samples_per_second': 142.932,
 'eval_steps_per_second': 8.978,
 'epoch': 3.0}